# OTP-스마트카드 유사도 검증
> 10개 OD pair 대상 6-Level 유사도 지표 + legGeometry 공간 유사도 검증

**검증 항목**:
1. 새 TCN 데이터 (정류장좌표시퀀스 포함) 확인
2. OTP legGeometry polyline 추출 검증
3. 레벨별 유사도 지표 통계 (mode / sequence / time / route / spatial)
4. 공간 유사도 변별력 (동일 vs 상이 노선/모드)
5. 가중치별 매칭 성공률/실패율
6. OD별 배정 결과

---
## 1. 설정

In [1]:
import pandas as pd
import numpy as np
import csv
import ijson
import os
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

In [2]:
%load_ext autoreload
%autoreload 2

from module.similarity import (
    parse_otp_itinerary,
    parse_smartcard_trip,
    deduplicate_itineraries,
    compute_all_metrics,
    compute_composite_similarity,
    compute_points_on_path_similarity,
    _extract_sc_known_coords,
    grade_similarity,
)
from module.route_features import fix_missing_distances

---
## 2. 데이터 로드

In [3]:
# 설정
OTP_INPUT_CSV = '../data/otp/input/otp_od_input_over13.csv'
OTP_JSON_PATH = '../data/otp/output/similarity.json'
TCN_PATH = '../data/tcn/20250217/TCN_20250217_route.parquet'
MAX_ODS = 10

# CSV에서 id -> od_pair 매핑
od_map = {}
with open(OTP_INPUT_CSV) as f:
    for i, row in enumerate(csv.DictReader(f)):
        od_map[i] = row['od_pair']
print(f'CSV OD pairs: {len(od_map):,}')

# OTP JSON 스트리밍 로드 (MAX_ODS개)
otp_results_raw = {}
loaded = 0
with open(OTP_JSON_PATH, 'rb') as f:
    for item in ijson.items(f, 'item', use_float=True):
        item_id = item.get('id')
        itins = item.get('data', {}).get('plan', {}).get('itineraries', [])
        if not itins:
            continue
        od_pair = od_map.get(item_id)
        if od_pair is None:
            continue
        for itin in itins:
            fix_missing_distances(itin)
        otp_results_raw[od_pair] = itins
        loaded += 1
        if loaded >= MAX_ODS:
            break

# 중복 제거 (>=2 대안)
otp_results = {}
for od_pair, itins in otp_results_raw.items():
    deduped = deduplicate_itineraries(itins)
    if len(deduped) >= 2:
        otp_results[od_pair] = deduped

print(f'OTP 로드: {loaded} ODs')
print(f'중복 제거 후 (>=2 대안): {len(otp_results)} ODs')
for od, itins in otp_results.items():
    print(f'  {od}: {len(itins)}개 대안')

CSV OD pairs: 774,895
OTP 로드: 10 ODs
중복 제거 후 (>=2 대안): 10 ODs
  10003_10661: 3개 대안
  10003_10700: 3개 대안
  10003_1451: 4개 대안
  10003_1457: 5개 대안
  10003_8001060: 4개 대안
  10003_8001754: 3개 대안
  10003_8001755: 2개 대안
  10003_8001757: 2개 대안
  10003_8001758: 3개 대안
  10003_8001761: 3개 대안


In [ ]:
# TCN 로드 (새 버전: 정류장좌표시퀀스 포함)
tcn = pd.read_parquet(TCN_PATH)
print(f'TCN 통행: {len(tcn):,}건')

# 새 컬럼 확인
has_coord_seq = '정류장lat시퀀스' in tcn.columns and '정류장lon시퀀스' in tcn.columns
print(f'정류장좌표시퀀스 존재: {has_coord_seq}')

if has_coord_seq:
    coord_lens = tcn['정류장lat시퀀스'].apply(len)
    print(f'좌표시퀀스 길이: 평균 {coord_lens.mean():.2f}, '
          f'2개(직통) {(coord_lens == 2).sum():,}, '
          f'3개+(환승) {(coord_lens >= 3).sum():,}')

# 테스트 대상 SC 추출
otp_od_set = set(otp_results.keys())
tcn_sample = tcn[tcn['od_pair'].isin(otp_od_set)]
print(f'\n테스트 대상 SC 통행: {len(tcn_sample):,}건')
print(tcn_sample['od_pair'].value_counts())

---
## 3. legGeometry polyline 추출 검증

In [ ]:
# OTP itinerary에서 route_polyline 추출 확인
for od_pair, itins in list(otp_results.items())[:3]:
    print(f'\nOD: {od_pair}')
    for i, itin in enumerate(itins):
        otp_parsed = parse_otp_itinerary(itin)
        polyline = otp_parsed.get('route_polyline', [])
        print(f'  경로 {i}: mode={otp_parsed["modes"]} routes={otp_parsed["routes"]}')
        print(f'    polyline: {len(polyline)} 좌표', end='')
        if polyline:
            lats = [p[0] for p in polyline]
            lons = [p[1] for p in polyline]
            print(f'  lat [{min(lats):.4f}, {max(lats):.4f}] lon [{min(lons):.4f}, {max(lons):.4f}]')
        else:
            print()

---
## 4. SC 환승 좌표 추출 검증

In [ ]:
# 환승 통행의 좌표 추출 확인
transfer_sample = tcn_sample[tcn_sample['환승횟수'] >= 1].head(10)
print(f'환승 통행 {len(transfer_sample)}건 샘플:\n')

for _, row in transfer_sample.iterrows():
    sc_parsed = parse_smartcard_trip(row)
    sc_coords = _extract_sc_known_coords(sc_parsed)
    print(f'OD: {row["od_pair"]}, 환승: {row["환승횟수"]}회')
    print(f'  정류장: {sc_parsed["stops"]}')
    print(f'  stop_coords(중간): {sc_parsed.get("stop_coords", [])}')
    print(f'  추출 좌표: {len(sc_coords)}개')
    print()

---
## 5. 전체 유사도 계산 (Non-GTFS, legGeometry 기반)

In [ ]:
# 가중치 설정
WEIGHTS = {
    'mode': 0.14, 'sequence': 0.21, 'time': 0.14,
    'route': 0.21, 'spatial': 0.30,
}
THRESHOLD = 0.6
print(f'가중치: {WEIGHTS}')
print(f'매칭 임계값: {THRESHOLD}')

In [ ]:
# 전체 SC 통행 vs OTP 대안 매칭
all_results = []

for od_pair in tqdm(otp_results.keys(), desc='유사도 계산'):
    itins = otp_results[od_pair]
    sc_trips = tcn_sample[tcn_sample['od_pair'] == od_pair]
    if len(sc_trips) == 0:
        continue

    for _, sc_row in sc_trips.iterrows():
        sc_parsed = parse_smartcard_trip(sc_row)
        sc_coords = _extract_sc_known_coords(sc_parsed)

        scores = []
        for idx, itin in enumerate(itins):
            otp_parsed = parse_otp_itinerary(itin)
            metrics = compute_all_metrics(otp_parsed, sc_parsed)
            cs = compute_composite_similarity(metrics, weights=WEIGHTS)
            scores.append({
                'idx': idx,
                'metrics': metrics,
                **cs,
                'route_match': 1 if set(otp_parsed['routes']) & set(sc_parsed['routes']) else 0,
                'mode_match': 1 if otp_parsed['modes'] == sc_parsed['modes'] else 0,
            })

        best = max(scores, key=lambda x: x['composite'])

        all_results.append({
            'od_pair': od_pair,
            'sc_category': sc_row.get('transport_category', ''),
            'sc_transfers': sc_parsed['transfer_count'],
            'sc_coords_count': len(sc_coords),
            'matched': best['composite'] >= THRESHOLD,
            'best_composite': round(best['composite'], 4),
            'best_idx': best['idx'],
            'mode_score': round(best['mode_score'], 4),
            'sequence_score': round(best['sequence_score'], 4),
            'time_score': round(best['time_score'], 4),
            'route_score': round(best['route_score'], 4),
            'spatial_score': round(best['spatial_score'], 4),
            'best_route_match': best['route_match'],
            'best_mode_match': best['mode_match'],
        })

results_df = pd.DataFrame(all_results)
print(f'총 SC 통행: {len(results_df):,}건')

---
## 6. 매칭 성공률 / 실패율

In [ ]:
total = len(results_df)
matched = results_df['matched'].sum()
failed = total - matched

print(f'=== 매칭 결과 (threshold={THRESHOLD}) ===')
print(f'총 SC 통행: {total:,}')
print(f'매칭 성공:  {matched:,} ({matched/total*100:.1f}%)')
print(f'매칭 실패:  {failed:,} ({failed/total*100:.1f}%)')

print(f'\n=== 등급 분포 ===')
results_df['grade'] = results_df['best_composite'].apply(grade_similarity)
print(results_df['grade'].value_counts().to_string())

print(f'\n=== 임계값별 매칭률 ===')
for th in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    n = (results_df['best_composite'] >= th).sum()
    print(f'  >= {th}: {n:,}/{total:,} ({n/total*100:.1f}%)')

---
## 7. 레벨별 유사도 통계

In [ ]:
level_cols = ['mode_score', 'sequence_score', 'time_score', 'route_score', 'spatial_score']

print('=== 레벨별 유사도 통계 (전체) ===')
print(results_df[level_cols + ['best_composite']].describe().round(4).to_string())

print(f'\n=== 레벨별 평균 ===')
for col in level_cols:
    mean = results_df[col].mean()
    std = results_df[col].std()
    print(f'  {col:20s}: {mean:.4f} (std={std:.4f})')
print(f'  {"best_composite":20s}: {results_df["best_composite"].mean():.4f} '
      f'(std={results_df["best_composite"].std():.4f})')

# 매칭 성공 vs 실패 비교
print(f'\n=== 매칭 성공 vs 실패 레벨별 평균 ===')
m = results_df[results_df['matched']]
f = results_df[~results_df['matched']]
print(f'{"":20s} {"성공(n="+str(len(m))+")":>15s} {"실패(n="+str(len(f))+")":>15s} {"차이":>10s}')
for col in level_cols + ['best_composite']:
    m_val = m[col].mean() if len(m) > 0 else 0
    f_val = f[col].mean() if len(f) > 0 else 0
    diff = m_val - f_val
    name = col.replace('_score', '').replace('best_', '')
    print(f'  {name:20s} {m_val:>14.4f} {f_val:>14.4f} {diff:>+10.4f}')

---
## 8. 공간 유사도 변별력 분석

In [ ]:
# SC 좌표 수 분포
print('=== SC 좌표 수 분포 ===')
print(results_df['sc_coords_count'].value_counts().sort_index().to_string())

# 전체 공간 유사도 통계
print(f'\n=== 공간 유사도 (spatial_score) 통계 ===')
print(results_df['spatial_score'].describe().round(4).to_string())

# 변별력: 노선 일치 vs 불일치
# best가 아닌 전체 대안 비교가 필요 → 별도 계산
print(f'\n--- 전체 대안 비교 변별력 ---')
pair_scores = []
for od_pair in otp_results.keys():
    itins = otp_results[od_pair]
    sc_trips = tcn_sample[tcn_sample['od_pair'] == od_pair]
    for _, sc_row in sc_trips.head(10).iterrows():
        sc_parsed = parse_smartcard_trip(sc_row)
        for idx, itin in enumerate(itins):
            otp_parsed = parse_otp_itinerary(itin)
            metrics = compute_all_metrics(otp_parsed, sc_parsed)
            pair_scores.append({
                'spatial': metrics.get('polyline_similarity', 0),
                'route_match': 1 if set(otp_parsed['routes']) & set(sc_parsed['routes']) else 0,
                'mode_match': 1 if otp_parsed['modes'] == sc_parsed['modes'] else 0,
                'sc_coords': len(_extract_sc_known_coords(sc_parsed)),
            })

pair_df = pd.DataFrame(pair_scores)
print(f'총 비교 쌍: {len(pair_df):,}')

same_r = pair_df[pair_df['route_match'] == 1]['spatial']
diff_r = pair_df[pair_df['route_match'] == 0]['spatial']
print(f'\n[노선 기반 변별력]')
print(f'  동일 노선: mean={same_r.mean():.4f} (n={len(same_r)})')
print(f'  상이 노선: mean={diff_r.mean():.4f} (n={len(diff_r)})')
if len(same_r) > 0 and len(diff_r) > 0:
    print(f'  차이: {same_r.mean() - diff_r.mean():.4f}')

same_m = pair_df[pair_df['mode_match'] == 1]['spatial']
diff_m = pair_df[pair_df['mode_match'] == 0]['spatial']
print(f'\n[모드 기반 변별력]')
print(f'  동일 모드: mean={same_m.mean():.4f} (n={len(same_m)})')
print(f'  상이 모드: mean={diff_m.mean():.4f} (n={len(diff_m)})')
if len(same_m) > 0 and len(diff_m) > 0:
    print(f'  차이: {same_m.mean() - diff_m.mean():.4f}')

# 환승 통행만
transfer_pairs = pair_df[pair_df['sc_coords'] >= 3]
if len(transfer_pairs) > 0:
    same_rt = transfer_pairs[transfer_pairs['route_match'] == 1]['spatial']
    diff_rt = transfer_pairs[transfer_pairs['route_match'] == 0]['spatial']
    print(f'\n[환승 통행만 (좌표 3개+) 노선 변별력]')
    print(f'  동일 노선: mean={same_rt.mean():.4f} (n={len(same_rt)})')
    print(f'  상이 노선: mean={diff_rt.mean():.4f} (n={len(diff_rt)})')
    if len(same_rt) > 0 and len(diff_rt) > 0:
        print(f'  차이: {same_rt.mean() - diff_rt.mean():.4f}')

---
## 9. OD별 매칭 결과

In [ ]:
# OD별 요약
od_summary = results_df.groupby('od_pair').agg(
    total=('matched', 'count'),
    matched=('matched', 'sum'),
    avg_composite=('best_composite', 'mean'),
    avg_mode=('mode_score', 'mean'),
    avg_sequence=('sequence_score', 'mean'),
    avg_time=('time_score', 'mean'),
    avg_route=('route_score', 'mean'),
    avg_spatial=('spatial_score', 'mean'),
).round(4)
od_summary['match_rate'] = (od_summary['matched'] / od_summary['total']).round(4)
od_summary['failed'] = od_summary['total'] - od_summary['matched']

print('=== OD별 매칭 결과 ===')
display_cols = ['total', 'matched', 'failed', 'match_rate',
                'avg_composite', 'avg_mode', 'avg_sequence',
                'avg_time', 'avg_route', 'avg_spatial']
print(od_summary[display_cols].to_string())

---
## 10. OD별 경로 배정 결과

In [ ]:
# OD별 OTP 경로 배정 상세
for od_pair in otp_results.keys():
    itins = otp_results[od_pair]
    od_sub = results_df[results_df['od_pair'] == od_pair]
    if len(od_sub) == 0:
        continue

    total = len(od_sub)
    matched_sub = od_sub[od_sub['matched']]
    unmatched_sub = od_sub[~od_sub['matched']]

    print(f'--- {od_pair} (SC {total}건) ---')

    # OTP 경로 요약 + 배정
    for i, itin in enumerate(itins):
        otp_p = parse_otp_itinerary(itin)
        n = len(matched_sub[matched_sub['best_idx'] == i])
        prob = n / total
        print(f'  [{i}] {sorted(otp_p["modes"])} {otp_p["routes"]} '
              f'환승={otp_p["transfer_count"]}회 '
              f'→ {n}건 ({prob*100:.1f}%)')

    n_other = len(unmatched_sub)
    print(f'  [other] → {n_other}건 ({n_other/total*100:.1f}%)')
    print()

---
## 11. 교통수단 카테고리별 매칭률

In [ ]:
cat_summary = results_df.groupby('sc_category').agg(
    total=('matched', 'count'),
    matched=('matched', 'sum'),
    avg_composite=('best_composite', 'mean'),
    avg_spatial=('spatial_score', 'mean'),
).round(4)
cat_summary['match_rate'] = (cat_summary['matched'] / cat_summary['total']).round(4)

print('=== 교통수단 카테고리별 매칭률 ===')
print(cat_summary.to_string())

# 환승 여부별
results_df['has_transfer'] = results_df['sc_transfers'] > 0
tr_summary = results_df.groupby('has_transfer').agg(
    total=('matched', 'count'),
    matched=('matched', 'sum'),
    avg_composite=('best_composite', 'mean'),
    avg_spatial=('spatial_score', 'mean'),
).round(4)
tr_summary['match_rate'] = (tr_summary['matched'] / tr_summary['total']).round(4)

print(f'\n=== 환승 여부별 매칭률 ===')
print(tr_summary.to_string())